<a href="https://colab.research.google.com/github/idcnys/flyrank/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb)


## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (Core lane).

I'm picking this lane because it turns directly into something a human can act on: a ranked list of pages to review first, with a stated reason for each one. The starter playground already ships an end-to-end version of this (feature prep, baseline score, model, ranked queue), which gives me an honest baseline to try to beat rather than starting from nothing. The output — a review queue with scores, actions, and reason codes — is also the most concrete of the four core lanes: it forces me to be explicit about what evidence backs each recommendation, instead of stopping at "this signal correlates with that outcome." That fits the way I want to spend the next 7 weeks: less time proving a single statistical point, more time building something a content reviewer could actually pick up and use.

## 2. The question: decision, action, cost of a wrong call

**The question:** Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring — given that a content/SEO reviewer only has time to look at a limited number of pages this cycle?

**Decision improved:** the order in which a content reviewer works through their backlog. Today that ordering likely comes from a hand-tuned rule or from whichever pages happen to get noticed. My work aims to replace or check that ordering with a ranked queue built from observed signals (visibility, freshness, position, engagement) plus reason codes explaining *why* each page is near the top.

**Who acts on it:** a content/SEO reviewer or strategist with fixed capacity — say, 20-50 pages they can realistically look at this cycle. They read the top of the queue, check the reason codes, and decide the actual content action (rewrite, expand, protect, prune, or leave alone).

**Cost of a wrong call, in both directions:**
- *False positive (flagged as worth reviewing, but it wasn't):* wastes a reviewer's limited time on a page that didn't need attention, and — repeated enough — erodes trust in the queue, so reviewers start ignoring it.
- *False negative (a genuinely at-risk or high-opportunity page never surfaces):* a page keeps quietly losing visibility or leaving clicks on the table until someone happens to notice it another way. Given fixed review capacity, missed pages are the more expensive failure mode, which is why precision@K *and* recall both matter here, not just overall accuracy.

Either way, the cost is a resourcing cost, not a safety cost — this is a decision-support ranking, not an automated action, so a wrong recommendation means wasted or delayed attention, not direct harm.

## 3. Quick look at the data (2-3 real numbers)


I'm loading `data/raw/content_refresh_anonymized.csv` and applying the same starter filter the pipeline uses (`impressions_90d > 0` and `content_age_days >= 90`, deduplicated by `content_id`), so the numbers below match what `scripts/01_prepare_features.py` would see.

In [2]:
import pandas as pd
from pathlib import Path

#running locally so i moved the data file
csv_path = Path("content_refresh_anonymized.csv")

if csv_path is None:
    raise FileNotFoundError(
        "Could not find content_refresh_anonymized.csv. "
        "Run this notebook from the repo (Colab badge above clones it for you), "
        "or update CSV_CANDIDATES with the correct path."
    )

raw = pd.read_csv(csv_path)
print(f"Loaded: {csv_path}")
print(f"Raw rows: {len(raw):,}  |  Raw columns: {len(raw.columns)}")

Loaded: content_refresh_anonymized.csv
Raw rows: 30,000  |  Raw columns: 44


In [3]:
required_cols = [
    "content_id", "client_id", "impressions_90d", "sessions_90d",
    "content_age_days", "trend_direction",
]
missing = [c for c in required_cols if c not in raw.columns]
assert not missing, f"Missing required columns: {missing}"

# Same row filter the starter pipeline applies before scoring anything.
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset="content_id")

print(f"Rows after filter (impressions_90d > 0, content_age_days >= 90): {len(df):,}")
print(f"Rows dropped by the filter: {len(raw) - len(df):,}")

Rows after filter (impressions_90d > 0, content_age_days >= 90): 30,000
Rows dropped by the filter: 0


In [4]:
# Three real numbers that motivate the lane.

n_clients = df["client_id"].nunique()
pct_declining = (df["trend_direction"] == "down").mean() * 100
median_impressions = df["impressions_90d"].median()

print(f"1. Content items in scope after filtering: {len(df):,}, across {n_clients} clients")
print(f"2. Share currently labeled 'declining' (trend_direction == 'down'): {pct_declining:.1f}%")
print(f"3. Median 90-day impressions among these pages: {median_impressions:,.0f}")

1. Content items in scope after filtering: 30,000, across 32 clients
2. Share currently labeled 'declining' (trend_direction == 'down'): 54.2%
3. Median 90-day impressions among these pages: 731


In [5]:
# One more cut that's specific to this lane: how many pages look like
# plausible review candidates under a couple of the starter's own reason-code rules,
# just from a quick look (not the final model) -- gives a sense of queue size.

if {"days_since_last_update", "word_count", "avg_position"}.issubset(df.columns):
    stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
    thin_visible = (
        (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)
    ).sum()
    print(f"4. 'stale_visible_page' candidates (>=180 days stale, >=500 impressions): {stale_visible:,}")
    print(f"5. 'thin_visible_page' candidates (<1200 words, >=250 impressions): {thin_visible:,}")
else:
    print("Optional columns (days_since_last_update / word_count / avg_position) not present in this slice.")

4. 'stale_visible_page' candidates (>=180 days stale, >=500 impressions): 17
5. 'thin_visible_page' candidates (<1200 words, >=250 impressions): 82


**What these numbers say about the lane:** even in the small anonymized starter slice, a meaningful share of in-scope pages are currently flagged as declining, and there's a real median volume of impressions behind them — meaning traffic worth protecting is actually at stake, not a handful of dead pages. The existence of several plausible reason-code candidates (stale-but-visible, thin-but-visible) even under simple rules suggests there's enough structure in the observable signals for a ranked queue to do better than "review whatever gets noticed." That's the gap this lane is meant to close, and the starter pipeline's own baseline-vs-model comparison (precision@50 of 0.240 for the rule-based baseline vs. 0.740 for a random forest, per `outputs/model_report.md`) is the evidence that a learned ranking is worth the next 7 weeks over a fixed rule.

## 4. Careful words: what I can and can't claim


**What this work will be able to say:**
- *Observed:* "In this snapshot, pages with X combination of signals were more often labeled declining / had lower CTR / had lower engagement than pages without it."
- *Directional:* "Pages with these characteristics tend to be worth reviewing sooner, based on historical patterns in the data."
- *Decision-support:* "This queue ranks pages by evidence and estimated opportunity so a reviewer with limited time reviews the most promising candidates first — it recommends where to look, not what will definitely happen."
- Every score will come with reason codes tracing back to observable signals (impressions, position, freshness, engagement), so a reviewer can check *why* a page ranked where it did, and disagree with it.

**What this work will never claim:**
- **No causal proof.** I will not say a refresh *caused* a recovery, or that not-refreshing *caused* a decline, unless I ran an actual experiment or causal design — which this data alone does not support.
- **No claims about Google's algorithm.** I will not claim to have discovered or proven a ranking factor Google uses; I only observe correlations between our own recorded signals and our own recorded outcomes.
- **No guarantee of results.** "High priority to review" is not the same as "guaranteed to recover if edited." A wrong recommendation costs reviewer time, not credibility in a promise I never made.
- **No reproduction of a product decision as if it were a discovery.** If I ever rebuild something like a `health_score` or `priority_score`-style rule, I'll present it explicitly as "can I reproduce the existing rule?" — not as new signal found in the data.
- **No private data exposure.** No client names, domains, raw URLs, titles, or queries will appear in any output, chart, or write-up — only

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.